# Lesson 24 — EC2 Production Deployment

## What you will learn
- **CloudWatch JSON logging** with structured fields (trace_id, tenant_id, host)
- **IAM instance profile** detection (no keys in code)
- **SSM Parameter Store** for secrets (not environment variables)
- **ALB health checks**: liveness vs readiness probes
- **Nginx** configuration for reverse proxy + SSE support
- **systemd** service for auto-restart and graceful shutdown
- Full **deployment checklist**

## Production stack
```
Internet
  ↓
ALB (Application Load Balancer)
  ↓  /health → liveness/readiness checks
Nginx (port 80/443)
  ↓  reverse proxy, SSE buffering off
Uvicorn (port 8000, 4 workers)
  ↓  FastAPI + LangGraph (L23 app)
systemd (manages uvicorn process)
```

## Security principles
```
❌  os.getenv("AWS_SECRET_KEY")         ← never hardcode/env secrets
✅  ssm.get_parameter("/p5/prod/key")   ← always use SSM in production
❌  jwt_secret = "my-secret"            ← never in source code
✅  IAM role for EC2 instance           ← automatic, no keys needed
```

In [ ]:
# !pip install boto3 watchtower

## Step 1 — CloudWatch JSON Formatter

In [ ]:
import json
import os
import socket
import logging
import time
from datetime import datetime

class CloudWatchJsonFormatter(logging.Formatter):
    """Emits structured JSON logs compatible with CloudWatch Insights."""
    SERVICE_NAME = os.getenv("SERVICE_NAME", "langgraph-api")
    HOSTNAME     = socket.gethostname()
    ENV          = os.getenv("ENV", "development")
    
    def format(self, record: logging.LogRecord) -> str:
        log = {
            "timestamp":  datetime.utcnow().isoformat() + "Z",
            "level":      record.levelname,
            "service":    self.SERVICE_NAME,
            "host":       self.HOSTNAME,
            "env":        self.ENV,
            "logger":     record.name,
            "message":    record.getMessage(),
        }
        # Inject extra fields (trace_id, tenant_id, etc.)
        for key in ["trace_id", "tenant_id", "user_id", "latency_ms", "status_code"]:
            if hasattr(record, key):
                log[key] = getattr(record, key)
        return json.dumps(log)

def setup_production_logging(log_group: str = "/p5/langgraph-api") -> logging.Logger:
    logger = logging.getLogger("p5")
    logger.setLevel(logging.INFO)
    
    # Console handler (always)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(CloudWatchJsonFormatter())
    logger.addHandler(console_handler)
    
    # CloudWatch handler (production)
    try:
        import watchtower, boto3
        cw_handler = watchtower.CloudWatchLogHandler(
            log_group=log_group,
            boto3_session=boto3.Session(),
        )
        cw_handler.setFormatter(CloudWatchJsonFormatter())
        logger.addHandler(cw_handler)
        print(f"CloudWatch logging enabled → {log_group}")
    except Exception:
        print("CloudWatch not available — console only (dev mode)")
    
    return logger

logger = setup_production_logging()

# Test structured log
record = logger.makeRecord("p5", logging.INFO, "", 0, "Request handled", (), None)
record.trace_id = "abc-123"
record.tenant_id = "acme"
record.latency_ms = 245.3
logger.handle(record)

## Step 2 — IAM Role Detection & SSM Config

In [ ]:
import urllib.request

def verify_iam_role_available() -> bool:
    """Check if EC2 instance metadata is reachable (= we're on EC2 with an IAM role)."""
    try:
        urllib.request.urlopen("http://169.254.169.254/latest/meta-data/", timeout=1)
        return True
    except Exception:
        return False

def load_config_from_ssm(param_path: str, default: str = "") -> str:
    """Load a secret from SSM Parameter Store, fall back to env var."""
    try:
        import boto3
        ssm = boto3.client("ssm")
        resp = ssm.get_parameter(Name=param_path, WithDecryption=True)
        return resp["Parameter"]["Value"]
    except Exception:
        # Fall back to env var — useful in dev without SSM
        env_key = param_path.split("/")[-1].upper()
        return os.getenv(env_key, default)

def load_production_config() -> dict:
    """Load all secrets — SSM first, env var fallback."""
    env = os.getenv("ENV", "development")
    return {
        "jwt_secret":    load_config_from_ssm(f"/p5/{env}/JWT_SECRET_KEY",  "dev-secret"),
        "s3_bucket":     load_config_from_ssm(f"/p5/{env}/S3_BUCKET_NAME",  "dev-bucket"),
        "bedrock_model": load_config_from_ssm(f"/p5/{env}/BEDROCK_MODEL_ID", "anthropic.claude-3-haiku-20240307-v1:0"),
        "oracle_dsn":    load_config_from_ssm(f"/p5/{env}/ORACLE_DSN",       ""),
    }

on_ec2 = verify_iam_role_available()
print(f"Running on EC2: {on_ec2}")

config = load_production_config()
print(f"Config loaded:")
for k, v in config.items():
    print(f"  {k}: {'*****' if 'secret' in k.lower() else v[:30]}")

## Step 3 — Health Checks

In [ ]:
def check_bedrock_connectivity() -> dict:
    try:
        import boto3
        boto3.client("bedrock", region_name="us-east-1").list_foundation_models()
        return {"bedrock": "ok"}
    except Exception as e:
        return {"bedrock": f"unavailable: {str(e)[:40]}"}

def check_s3_connectivity(bucket: str = "dev-bucket") -> dict:
    try:
        import boto3
        boto3.client("s3").head_bucket(Bucket=bucket)
        return {"s3": "ok"}
    except Exception as e:
        return {"s3": f"unavailable: {str(e)[:40]}"}

def get_liveness() -> dict:
    """ALB liveness probe — is the process alive? Always 200 if running."""
    return {"status": "alive", "ts": datetime.utcnow().isoformat()}

def get_readiness() -> dict:
    """ALB readiness probe — can we serve traffic? Check dependencies."""
    checks = {}
    # Bedrock and S3 only checked in production
    if os.getenv("ENV", "development") == "production":
        checks.update(check_bedrock_connectivity())
        checks.update(check_s3_connectivity())
    
    all_ok = all(v == "ok" for v in checks.values()) if checks else True
    return {
        "status": "ready" if all_ok else "not_ready",
        "checks": checks,
        "ts": datetime.utcnow().isoformat(),
    }

print("Liveness:",  json.dumps(get_liveness(), indent=2))
print("Readiness:", json.dumps(get_readiness(), indent=2))

## Step 4 — Nginx Config Generator

In [ ]:
def generate_nginx_config(server_name: str = "api.example.com", upstream_port: int = 8000) -> str:
    return f"""# /etc/nginx/conf.d/langgraph-api.conf
upstream langgraph {{
    server 127.0.0.1:{upstream_port};
    keepalive 32;
}}

server {{
    listen 80;
    server_name {server_name};

    # SSE: disable buffering so tokens stream immediately
    proxy_buffering off;
    proxy_read_timeout 120s;
    proxy_send_timeout 120s;

    location / {{
        proxy_pass http://langgraph;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Request-ID $request_id;
    }}

    location /health {{
        proxy_pass http://langgraph/health;
        access_log off;
    }}
}}
"""

print(generate_nginx_config())

## Step 5 — systemd Service Generator

In [ ]:
def generate_systemd_service(app_dir: str = "/opt/p5", workers: int = 4) -> str:
    return f"""# /etc/systemd/system/langgraph-api.service
[Unit]
Description=LangGraph API Service
After=network.target
Wants=network-online.target

[Service]
Type=simple
User=p5
Group=p5
WorkingDirectory={app_dir}
EnvironmentFile=/etc/p5/.env
ExecStart={app_dir}/.venv/bin/uvicorn \\
    lesson_23_conversation_api.lesson_23_conversation_api:app \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --workers {workers} \\
    --timeout-keep-alive 30

Restart=always
RestartSec=5
StandardOutput=journal
StandardError=journal
SyslogIdentifier=langgraph-api

# Resource limits
LimitNOFILE=65536
MemoryMax=4G

[Install]
WantedBy=multi-user.target
"""

print(generate_systemd_service())

## Step 6 — Deployment Checklist

In [ ]:
DEPLOYMENT_CHECKLIST = """
EC2 PRODUCTION DEPLOYMENT CHECKLIST
=====================================

PRE-DEPLOY
[ ] All secrets stored in SSM Parameter Store (/p5/prod/*)
[ ] EC2 instance has IAM role with SSM + S3 + Bedrock permissions
[ ] Security group: inbound 80/443 only from ALB, outbound all
[ ] Python 3.11+ installed, virtualenv created
[ ] requirements.txt installed in .venv

DEPLOY
[ ] git pull latest code to /opt/p5
[ ] systemctl daemon-reload
[ ] systemctl restart langgraph-api
[ ] systemctl status langgraph-api  (check no crash loops)

POST-DEPLOY
[ ] curl http://localhost:8000/health  → {"status": "alive"}
[ ] curl http://localhost:8000/readiness → {"status": "ready"}
[ ] Check ALB target group shows instances as healthy
[ ] Check CloudWatch log group /p5/langgraph-api for JSON logs
[ ] Run smoke test: POST /sessions + POST /chat
[ ] Monitor p95 latency in CloudWatch for 15 minutes

ROLLBACK (if needed)
[ ] systemctl stop langgraph-api
[ ] git checkout <previous-tag>
[ ] systemctl start langgraph-api
"""
print(DEPLOYMENT_CHECKLIST)

## Key Takeaways

| Component | Production pattern |
|-----------|-------------------|
| Logging | `CloudWatchJsonFormatter` → stdout → CW agent |
| Secrets | SSM Parameter Store → `load_config_from_ssm()` |
| Auth | IAM instance profile → no AWS keys in code |
| Health | `/health` liveness + `/readiness` dependency check |
| Nginx | `proxy_buffering off` for SSE; `proxy_read_timeout 120s` |
| systemd | `Restart=always`, `EnvironmentFile=/etc/p5/.env` |

## 🏋️ Exercise
1. Add a `/metrics` endpoint returning current server stats: uptime, request count, error rate
2. Generate a complete `launch_template.json` for EC2 auto-scaling with user data script that runs the deployment steps
3. Write a `pre_flight_check()` function that runs ALL health checks before `app.startup` and refuses to start if any critical dependency fails

In [ ]:
# Your exercise solution here
